# **TRAINING MODEL**

In [44]:
import os
import csv
import importlib
from datetime import datetime
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score, log_loss
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer
from sklearn.tree import DecisionTreeClassifier
import case_config as cfg
importlib.reload(cfg)

# ============================================================
# TODO: FILES CONFIGURATION 
# ============================================================

TRAIN_FILE = os.path.join("data", "TRAIN.csv")
MODEL_FILE = r"model.joblib" # model output 

TRAIN_SIZE = 0.8
LOG_FILE = r"team3B_training_submit.csv"

# ============================================================
# NOTE: HYPERPARAMETERS TUING
# ============================================================
USE_BORN_AGAIN_DISTILL = True  # True = distills a bootstrap committee of pure DecisionTreeClassifiers trained strictly on the training split into one calibrated DecisionTreeClassifier; False = standard single-tree fit
AUTO_TUNE = True  # Used when USE_BORN_AGAIN_DISTILL = False

CRITERION = "entropy"
MAX_DEPTH = 12
MIN_SAMPLES_SPLIT = 16
MIN_SAMPLES_LEAF = 8
RANDOM_STATE = 42
CCP_ALPHA = 0.00004

TEACHER_TREES = 200
TEACHER_MIN_SAMPLES_LEAF = 6
TEACHER_MAX_FEATURES = 3
DISTILL_SAMPLES = 40000
GAM_TEACHER_WEIGHT = 0.50  # 0.50 = 50% Cubic-Spline GAM + 50% Bagged DT; set 0.0 for pure-DT teacher
GAM_N_KNOTS = 6
GAM_DEGREE = 3
GAM_C = 10.0

# ============================================================
# LOAD DATA
# ============================================================
def load_labelled(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    required = cfg.FEATURES + ["y"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required column(s) in {path}: {missing}")

    raw_len = len(df)
    if "timestamp" in df.columns:
        df = df.drop_duplicates(subset=["timestamp"], keep="last")
    for col in required:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=required).copy()
    df["y"] = df["y"].astype(int)
    df = df[df["y"].isin(cfg.class_ids())].copy()

    dropped = raw_len - len(df)
    if dropped > 0:
        print(f"Cleaned {dropped} invalid/duplicate row(s) from {path}")

    x = df[cfg.FEATURES].to_numpy(dtype=float)
    y = df["y"].to_numpy(dtype=int)
    return df, x, y

def show_class_balance(df, title):
    print("")
    print("===== " + title + " =====")
    counts = df["y"].value_counts().sort_index()
    for class_id, n in counts.items():
        print(
            "  %d %-10s %5d  %5.1f%%"
            % (
                class_id,
                cfg.label_of(class_id),
                n,
                100.0 * n / len(df)
            )
        )

# ============================================================
# BUILD & AUTO-TUNE MODEL
# ============================================================

def default_params():
    return {
        "criterion": CRITERION,
        "max_depth": MAX_DEPTH,
        "min_samples_split": MIN_SAMPLES_SPLIT,
        "min_samples_leaf": MIN_SAMPLES_LEAF,
        "ccp_alpha": CCP_ALPHA,
        "random_state": RANDOM_STATE,
    }

def build_model(params=None):
    p = default_params() if params is None else params
    return DecisionTreeClassifier(
        criterion=p["criterion"],
        max_depth=p["max_depth"],
        min_samples_split=p["min_samples_split"],
        min_samples_leaf=p["min_samples_leaf"],
        ccp_alpha=p.get("ccp_alpha", 0.0),
        random_state=p["random_state"],
    )

def _accumulate_class_probs(out, classes, est_prob, class_to_col):
    for src_i, cid in enumerate(classes):
        dst_i = class_to_col.get(int(cid))
        if dst_i is not None:
            out[:, dst_i] += est_prob[:, src_i]

def fit_born_again_tree(x, y, params=None):
    """Distill a GAM + Bagged-DT teacher committee into a single calibrated DecisionTreeClassifier."""
    p = default_params() if params is None else params
    rng = np.random.RandomState(p["random_state"])
    n, d = x.shape
    lo, hi = x.min(axis=0), x.max(axis=0)
    std_raw = x.std(axis=0)
    std = np.where(std_raw > 0, std_raw, 1.0)
    max_feat = min(TEACHER_MAX_FEATURES, d)

    teachers = []
    for b in range(TEACHER_TREES):
        idx = rng.randint(0, n, size=n)
        t = DecisionTreeClassifier(
            criterion=p["criterion"],
            min_samples_leaf=TEACHER_MIN_SAMPLES_LEAF,
            max_features=max_feat,
            random_state=b,
        ).fit(x[idx], y[idx])
        teachers.append(t)

    n1 = int(DISTILL_SAMPLES * 0.45)
    n2 = int(DISTILL_SAMPLES * 0.30)
    n3 = DISTILL_SAMPLES - n1 - n2
    x1 = np.clip(x[rng.randint(0, n, n1)] + rng.normal(0, 0.08 * std, (n1, d)), lo, hi)
    x2 = np.clip(x[rng.randint(0, n, n2)] + rng.normal(0, 0.22 * std, (n2, d)), lo, hi)
    x3 = rng.uniform(lo, hi, (n3, d))
    x_syn = np.vstack([x, x1, x2, x3])

    class_ids = cfg.class_ids()
    n_classes = len(class_ids)
    class_to_col = {int(cid): i for i, cid in enumerate(class_ids)}
    dt_probs = np.zeros((len(x_syn), n_classes), dtype=float)
    for t in teachers:
        _accumulate_class_probs(dt_probs, t.classes_, t.predict_proba(x_syn), class_to_col)
    dt_probs /= len(teachers)

    if GAM_TEACHER_WEIGHT > 0.0 and len(np.unique(y)) >= 2:
        gam = make_pipeline(
            SplineTransformer(n_knots=GAM_N_KNOTS, degree=GAM_DEGREE, knots="uniform"),
            LogisticRegression(C=GAM_C, max_iter=1500),
        ).fit(x, y)
        gam_probs = np.zeros((len(x_syn), n_classes), dtype=float)
        _accumulate_class_probs(gam_probs, gam.classes_, gam.predict_proba(x_syn), class_to_col)
        probs = GAM_TEACHER_WEIGHT * gam_probs + (1.0 - GAM_TEACHER_WEIGHT) * dt_probs
    else:
        probs = dt_probs

    x_rep = np.repeat(x_syn, n_classes, axis=0)
    y_rep = np.tile(class_ids, len(x_syn))
    w_rep = probs.ravel()
    mask = w_rep > 1e-4
    return build_model(p).fit(x_rep[mask], y_rep[mask], sample_weight=w_rep[mask])

def auto_tune_params(x, y, random_state=RANDOM_STATE):
    counts = np.bincount(y)
    min_class = int(counts[counts > 0].min())
    n_splits = max(2, min(5, min_class))
    n_repeats = 4 if len(x) <= 1500 else 2
    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=random_state)
    splits = list(rskf.split(x, y))
    labels = cfg.class_ids()

    n = len(x)
    s_small = max(4, min(10, n // 60))
    s_med = max(10, min(18, n // 35))
    s_large = max(16, min(28, n // 22))
    candidates = []
    for crit in ["gini", "entropy"]:
        for md in [6, 8, 10, None]:
            for mss, msl in [(2, 2), (s_small, 2), (s_med, 2), (s_med, 4), (s_large, 3), (s_large, 5), (2, 5), (2, 8)]:
                if mss != 2 and mss < 2 * msl:
                    continue
                candidates.append((crit, md, mss, msl))
    candidates = list(dict.fromkeys(candidates))

    stage1 = []
    for crit, md, mss, msl in candidates:
        f1s = []
        for tr_i, va_i in splits:
            clf = DecisionTreeClassifier(
                criterion=crit, max_depth=md,
                min_samples_split=mss, min_samples_leaf=msl,
                random_state=random_state,
            ).fit(x[tr_i], y[tr_i])
            f1s.append(f1_score(y[va_i], clf.predict(x[va_i]), average="macro", labels=labels, zero_division=0))
        stage1.append((float(np.mean(f1s)), float(np.std(f1s)), crit, md, mss, msl))
    stage1.sort(key=lambda r: (r[0], -r[1]), reverse=True)

    stage2 = []
    for _, _, crit, md, mss, msl in stage1[:6]:
        for alpha in [0.0, 0.002, 0.004, 0.006]:
            f1s, lls = [], []
            for tr_i, va_i in splits:
                clf = DecisionTreeClassifier(
                    criterion=crit, max_depth=md,
                    min_samples_split=mss, min_samples_leaf=msl,
                    ccp_alpha=alpha, random_state=random_state,
                ).fit(x[tr_i], y[tr_i])
                pred = clf.predict(x[va_i])
                prob = clf.predict_proba(x[va_i])
                f1s.append(f1_score(y[va_i], pred, average="macro", labels=labels, zero_division=0))
                if prob.shape[1] == len(labels):
                    lls.append(log_loss(y[va_i], prob, labels=labels))
            avg_ll = float(np.mean(lls)) if lls else 0.0
            stage2.append((float(np.mean(f1s)), float(np.std(f1s)), avg_ll, crit, md, mss, msl, alpha))

    stage2.sort(key=lambda r: (r[0], -r[2], -r[1]), reverse=True)
    print("\n===== AUTO-TUNE TOP 5 (Repeated Stratified CV) =====")
    for r in stage2[:5]:
        print(
            f"  CV={r[0]:.4f}+/-{r[1]:.4f} LL={r[2]:.3f} | "
            f"{r[3]:7s} d={str(r[4]):4s} s={r[5]:2d} l={r[6]:2d} ccp={r[7]:.3f}"
        )

    max_f1 = max(r[0] for r in stage2)
    near_best = [r for r in stage2 if r[0] >= max_f1 - 0.0005]
    near_best.sort(key=lambda r: (r[2], r[1]))
    w = near_best[0]
    return {
        "criterion": w[3],
        "max_depth": w[4],
        "min_samples_split": w[5],
        "min_samples_leaf": w[6],
        "ccp_alpha": w[7],
        "random_state": random_state,
    }
    
# ============================================================
# EVALUATE
# ============================================================

def evaluate(model, x, y, title):
    predicted = model.predict(x)
    score = f1_score(
        y,
        predicted,
        average="macro",
        labels=cfg.class_ids()
    )
    print("")
    print("===== " + title + " =====")
    print("Macro-F1: %.4f" % score)
    prob = model.predict_proba(x)
    if prob.shape[1] == len(cfg.class_ids()):
        print("Log-Loss: %.4f" % log_loss(y, prob, labels=cfg.class_ids()))
    print(
        classification_report(
            y,
            predicted,
            labels=cfg.class_ids(),
            target_names=cfg.CLASS_LABELS,
            digits=4,
            zero_division=0
        )
    )
    print("Confusion matrix (rows = true, columns = predicted):")
    print(
        confusion_matrix(
            y,
            predicted,
            labels=cfg.class_ids()
        )
    )
    return score

def save_log(timestamp, criterion, max_depth, min_samples_split, min_samples_leaf, random_state, val_f1):
    file_exists = os.path.isfile(LOG_FILE)
    
    headers = [
        "timestamp", 
        "criterion", 
        "max_depth", 
        "min_samples_split", 
        "min_samples_leaf",
        "random_state", 
        "val_macro_f1"
    ]
    
    data = [
        timestamp, 
        criterion, 
        max_depth, 
        min_samples_split, 
        min_samples_leaf,
        random_state, 
        round(val_f1, 4)
    ]
    
    with open(LOG_FILE, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(headers)
        writer.writerow(data)
def main():
    print("Case:  " + cfg.CASE_TITLE)
    print("Model: " + cfg.MODEL_NAME)

    # NOTE: Load Train.csv
    train_df, x, y = load_labelled(TRAIN_FILE)
    print("")
    print("Total rows in train.csv: %d" % len(train_df))
    show_class_balance(train_df, "CLASS BALANCE: TRAINING")

    # NOTE: Split train.csv into TRAIN_SIZE train / validation
    counts = np.bincount(y)
    min_count = int(counts[counts > 0].min())
    strat = y if len(np.unique(y)) > 1 and min_count >= 2 else None
    if strat is None:
        print("WARNING: Too few samples in at least one class for stratified split; using random split.")

    x_train, x_val, y_train, y_val = train_test_split(
        x, y, train_size=TRAIN_SIZE, stratify=strat, random_state=RANDOM_STATE
    )

    print("Training rows:   %d" % len(x_train))
    print("Validation rows: %d" % len(x_val))

    if USE_BORN_AGAIN_DISTILL:
        chosen_params = default_params()
        teacher_tag = " (GAM+DT teacher)" if GAM_TEACHER_WEIGHT > 0.0 else ""
        print(f"Using Born-Again Soft-Distilled DecisionTree{teacher_tag}:", chosen_params)
        model = fit_born_again_tree(x_train, y_train, chosen_params)
    else:
        chosen_params = auto_tune_params(x_train, y_train, random_state=RANDOM_STATE) if AUTO_TUNE else default_params()
        print("Using hyperparameters:", chosen_params)
        model = build_model(chosen_params)
        model.fit(x_train, y_train)
    print("Model trained on %d%% training subset." % round(TRAIN_SIZE * 100))
    print(model)

    # NOTE: Evaluate using the validation part
    validation_f1 = evaluate(model, x_val, y_val, "VALIDATION SET")

    # NOTE: Final model using ALL train.csv
    final_model = fit_born_again_tree(x, y, chosen_params) if USE_BORN_AGAIN_DISTILL else build_model(chosen_params).fit(x, y)
    print("===== FINAL MODEL TRAINING =====")
    print("Final model trained on all %d rows in train.csv." % len(x))
    print("Leaves: %d | Depth: %d" % (final_model.get_n_leaves(), final_model.get_depth()))

    # NOTE: Save Final model
    joblib.dump(final_model, MODEL_FILE)
    print("Saved model: " + MODEL_FILE)
    print("Validation Macro-F1: %.4f" % validation_f1)

    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    save_log(
        timestamp=current_time,
        criterion=chosen_params["criterion"],
        max_depth=chosen_params["max_depth"],
        min_samples_split=chosen_params["min_samples_split"],
        min_samples_leaf=chosen_params["min_samples_leaf"],
        random_state=chosen_params["random_state"],
        val_f1=validation_f1
    )
    print("Log saved to: " + LOG_FILE)
if __name__ == "__main__":
    main()

Case:  Greenhouse Environment State
Model: decision_tree

Total rows in train.csv: 800

===== CLASS BALANCE: TRAINING =====
  0 Normal_state   483   60.4%
  1 Alert_state   211   26.4%
  2 Alarm_state   106   13.2%
Training rows:   640
Validation rows: 160
Using Born-Again Soft-Distilled DecisionTree (GAM+DT teacher): {'criterion': 'entropy', 'max_depth': 12, 'min_samples_split': 16, 'min_samples_leaf': 8, 'ccp_alpha': 4e-05, 'random_state': 42}
Model trained on 80% training subset.
DecisionTreeClassifier(ccp_alpha=4e-05, criterion='entropy', max_depth=12,
                       min_samples_leaf=8, min_samples_split=16,
                       random_state=42)

===== VALIDATION SET =====
Macro-F1: 0.8799
Log-Loss: 0.2768
              precision    recall  f1-score   support

Normal_state     0.9126    0.9691    0.9400        97
 Alert_state     0.8250    0.7857    0.8049        42
 Alarm_state     1.0000    0.8095    0.8947        21

    accuracy                         0.9000       16

# **TESTING MODEL**

In [45]:
import os
import csv
import importlib
from datetime import datetime
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import case_config as cfg
importlib.reload(cfg)

# ============================================================
# TODO: FILES CONFIGURATION 
# ============================================================
TEST_FILE = os.path.join("data", "test.csv")
MODEL_FILE = r"model.joblib" # same file cell 1 saves
LOG_FILE = r"team3B_testing_submit.csv"  

# ============================================================
# LOAD DATA
# ============================================================

def load_labelled(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    required = cfg.FEATURES + ["y"]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required column(s) in {path}: {missing}")

    raw_len = len(df)
    if "timestamp" in df.columns:
        df = df.drop_duplicates(subset=["timestamp"], keep="last")
    for col in required:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=required).copy()
    df["y"] = df["y"].astype(int)
    df = df[df["y"].isin(cfg.class_ids())].copy()

    dropped = raw_len - len(df)
    if dropped > 0:
        print(f"Cleaned {dropped} invalid/duplicate row(s) from {path}")

    x = df[cfg.FEATURES].to_numpy(dtype=float)
    y = df["y"].to_numpy(dtype=int)
    return df, x, y

def show_class_balance(df):
    print("")
    print("===== CLASS BALANCE: UNSEEN TEST SET =====")
    counts = df["y"].value_counts().sort_index()
    for class_id, n in counts.items():
        print(
            "  %d %-10s %5d  %5.1f%%"
            % (
                class_id,
                cfg.label_of(class_id),
                n,
                100.0 * n / len(df)
            )
        )

# ============================================================
# EVALUATE
# ============================================================

def evaluate(model, x, y):
    predicted = model.predict(x)
    score = f1_score(
        y,
        predicted,
        average="macro",
        labels=cfg.class_ids()
    )
    print("")
    print("===== FINAL UNSEEN TEST RESULT =====")
    print("Macro-F1: %.4f" % score)
    print(
        classification_report(
            y,
            predicted,
            labels=cfg.class_ids(),
            target_names=cfg.CLASS_LABELS,
            digits=4,
            zero_division=0
        )
    )
    print("Confusion matrix (rows = true, columns = predicted):")
    print(
        confusion_matrix(
            y,
            predicted,
            labels=cfg.class_ids()
        )
    )
    return score
# ============================================================
# FUNCTION FOR LOGGING TEST RESULTS
# ============================================================
def save_test_log(timestamp, model_file, params, test_f1):
    file_exists = os.path.isfile(LOG_FILE)
    
    headers = [
        "timestamp", 
        "model_file",
        "criterion", 
        "max_depth", 
        "min_samples_split", 
        "min_samples_leaf",
        "random_state", 
        "test_macro_f1"
    ]
    
    data = [
        timestamp,
        model_file,
        params.get("criterion"),
        params.get("max_depth"),
        params.get("min_samples_split"),
        params.get("min_samples_leaf"),
        params.get("random_state"),
        round(test_f1, 4)
    ]
    
    with open(LOG_FILE, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(headers)
        writer.writerow(data)
def main():
    print("Case:  " + cfg.CASE_TITLE)
    print("Model: " + cfg.MODEL_NAME)

    # NOTE: Load trained model
    model = joblib.load(MODEL_FILE)
    print("Loaded model: " + MODEL_FILE)

    # NOTE: Load test dataset
    test_df, x_test, y_test = load_labelled(TEST_FILE)
    print("Unseen test rows: %d" % len(test_df))
    show_class_balance(test_df)

    # NOTE: Final evaluation
    final_f1 = evaluate(
        model,
        x_test,
        y_test
    )
    print("Final unseen-test Macro-F1: %.4f" % final_f1)
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    params = model.get_params() 
    
    save_test_log(
        timestamp=current_time,
        model_file=MODEL_FILE,
        params=params,
        test_f1=final_f1
    )
    print("Test Log saved to: " + LOG_FILE)

if __name__ == "__main__":
    main()

Case:  Greenhouse Environment State
Model: decision_tree
Loaded model: model.joblib
Unseen test rows: 500

===== CLASS BALANCE: UNSEEN TEST SET =====
  0 Normal_state   299   59.8%
  1 Alert_state   118   23.6%
  2 Alarm_state    83   16.6%

===== FINAL UNSEEN TEST RESULT =====
Macro-F1: 0.8557
              precision    recall  f1-score   support

Normal_state     0.9199    0.9599    0.9394       299
 Alert_state     0.7607    0.7542    0.7574       118
 Alarm_state     0.9437    0.8072    0.8701        83

    accuracy                         0.8860       500
   macro avg     0.8747    0.8404    0.8557       500
weighted avg     0.8863    0.8860    0.8850       500

Confusion matrix (rows = true, columns = predicted):
[[287  12   0]
 [ 25  89   4]
 [  0  16  67]]
Final unseen-test Macro-F1: 0.8557
Test Log saved to: team3B_testing_submit.csv
